In [ ]:
library(Seurat)
library(ggplot2)
library(ggpubr)
library(pheatmap)
library(dplyr)
library(RColorBrewer)
getwd()
dir.create("figures_10xMouse_RA")
dir.create("data_10xMouse_RA")
colorDict = c("Differentiating"="#88535A",
              "2CLC"="#EF8264",
              "Pluripotent"="#F2CC8F")


dataset_id <- "10xMouse_RA"
sample <- "LIF"

In [ ]:
dataset_id <- "10xMouse_RA"
sample <- "LIF"
path <- paste0("/mnt/TEresults2/snakemake_results/results/stellarscope_out/", dataset_id, "/", sample, "/pseudobulk/")


STAR_path <- "/mnt/TEresults/snakemake_results/results/STARoutdir/10xMouse_RA/LIF/best_Solo.out/Gene"
filteredBarcodes <- read.table(paste0(STAR_path, "/filtered/barcodes.tsv"))$V1 # read barcodes seleced by STARsolo


twoCLC_TEmatrix <- Seurat::ReadMtx(mtx = paste0(path, sample, "_pseudobulk-TE_counts.mtx"), 
                              cells = paste0(path, sample, "_pseudobulk-barcodes.tsv"), 
                              features = paste0(path, sample, "_pseudobulk-features.tsv"), 
                              feature.column = 1) # read matrix
twoCLC_TEmatrix <- twoCLC_TEmatrix[setdiff(rownames(twoCLC_TEmatrix),"__no_feature"),filteredBarcodes]

nCells <- ncol(twoCLC_TEmatrix)
thrMinCells <- round(nCells * 0.02)
thrMinCells

# create Seurat object with shallow filtering of TEs expressed in at least 20 cells and cells expressing at least 50 TEs 
objTE_stellarscope <- Seurat::CreateSeuratObject(twoCLC_TEmatrix, project = "2CLC", 
                          min.cells = thrMinCells, min.features = 50)

#objTE_stellarscope <- objTE_allCB[,filteredBarcodes] # keep only filtered barcodes

objTE_stellarscope

In [ ]:
summary(rowSums(objTE_stellarscope@assays$RNA$counts > 0))

In [ ]:
annotation_stellarscope <- read.table("annotation/annotation_stellarscope.tsv") # generated with annotation_scripts/create_annotations_mouse.Rmd
head(annotation_stellarscope)

In [ ]:
# Remove some classes of TEs

classesToExclude <- c("Other", "Satellite", "Unknown", "RNA")

stellarscopeTEs <- Features(objTE_stellarscope)

TEsToKeep <- annotation_stellarscope[! annotation_stellarscope$class %in% classesToExclude, ]$stellarscopeID

length(setdiff(stellarscopeTEs, TEsToKeep))/ length(stellarscopeTEs) * 100 # percentage removed

objTE_stellarscope <- objTE_stellarscope[intersect(stellarscopeTEs, TEsToKeep),]

In [ ]:
feature_metadata <- annotation_stellarscope[match(Features(objTE_stellarscope), annotation_stellarscope$stellarscopeID),]

head(feature_metadata)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)


objTE_stellarscope@meta.data$nCount_TE <- objTE_stellarscope@meta.data$nCount_RNA 
objTE_stellarscope@meta.data$nFeature_TE <- objTE_stellarscope@meta.data$nFeature_RNA 
# Visualize QC metrics as a violin plot
VlnPlot(objTE_stellarscope, features = c("nCount_TE"), ncol = 1, 
        cols = colorDict, pt.size = 0) + theme(text=element_text(size=17))

ggsave("figures_10xMouse_RA/nCountTE_violin_stellarscope.png", device='png',dpi=600)
ggsave("figures_10xMouse_RA/nCountTE_violin_stellarscope.pdf", device='pdf')

VlnPlot(objTE_stellarscope, features = c("nFeature_TE"), ncol = 1, 
        cols = colorDict, pt.size = 0) + theme(text=element_text(size=17))
ggsave("figures_10xMouse_RA/nFeatureTE_violin_stellarscope.png", device='png',dpi=600)
ggsave("figures_10xMouse_RA/nFeatureTE_violin_stellarscope.pdf", device='pdf')


In [ ]:

objTE_stellarscope <- JoinLayers(objTE_stellarscope)

objTE_stellarscope <- NormalizeData(objTE_stellarscope, normalization.method = "LogNormalize", scale.factor = 10000)

objTE_stellarscope <- FindVariableFeatures(objTE_stellarscope, selection.method = "vst", nfeatures = 4000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objTE_stellarscope), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objTE_stellarscope)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:
objTE_stellarscope

In [ ]:

gc()
all.genes <- rownames(objTE_stellarscope)
objTE_stellarscope <- ScaleData(objTE_stellarscope) # on hvgs


In [ ]:
grep("MERVL", all.genes, value = T)[1:20]


In [ ]:

objTE_stellarscope <- RunPCA(objTE_stellarscope, features = VariableFeatures(object = objTE_stellarscope))

DimPlot(objTE_stellarscope, reduction = "pca") + NoLegend()

ElbowPlot(objTE_stellarscope)


In [ ]:
objTE_stellarscope <- FindNeighbors(objTE_stellarscope, dims = 1:15, k.param = 20)
objTE_stellarscope <- FindClusters(objTE_stellarscope, resolution = 1, algorithm=4)
objTE_stellarscope <- RunUMAP(objTE_stellarscope, dims = 1:15)
DimPlot(objTE_stellarscope, reduction = "umap")


In [ ]:
FeaturePlot(objTE_stellarscope, reduction = "umap", features = "nCount_TE", pt.size = 0.5) + 
  theme_void() +
  theme(text=element_text(size=20))

In [ ]:

FeaturePlot(objTE_stellarscope, features = c('MERVL-int-dup60','MERVL-int-dup5'), pt.size=1,alpha = 0.8,order = TRUE ) & 
  theme_void() &
  theme(text=element_text(size=20)) 

In [ ]:

#saveRDS(objTE_stellarscope, paste0("data_", dataset_id, "/stellarscope_", dataset_id, "_seuratObj_251125.RDS"))
